# ML-04 — Refresh / Content Opportunity Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardikkk-1209/ML_Pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This contract defines what each row means, which fields are safe features, what is the label/proxy, and what is excluded.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Unique content pages:", f"{df['content_id'].nunique():,}")
print("Duplicate content IDs:", int(df['content_id'].duplicated().sum()))
print("This starter snapshot contains trailing-90-day page-level metrics.")

**Unit of analysis:** one row represents one pseudonymized content page.

**Time window:** the starter metrics such as `impressions_90d`, `clicks_90d`, and `sessions_90d` summarize the trailing 90 days represented by the snapshot. The starter CSV has no report-date column, so I will not invent a calendar date for it.

For the full warehouse, the unit becomes one `report_date + client_hash_id + content_hash_id` row and feature/outcome windows must be kept separate.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
feature_fields = [
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr",
    "avg_position", "content_age_days", "days_since_last_update",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
context_fields = ["content_id", "client_id", "content_type", "main_intent"]
label_proxy = ["is_declining_label", "trend_direction"]
excluded_fields = ["trend_pct", "health_score", "needs_indexing", "needs_ctr_fix",
                   "needs_engagement_fix", "is_quick_win", "ai_opportunity",
                   "is_underperformer", "is_declining"]

print("Features:", [c for c in feature_fields if c in df.columns])
print("Context:", [c for c in context_fields if c in df.columns])
print("Label/proxy source:", [c for c in label_proxy if c in df.columns])
print("Excluded:", [c for c in excluded_fields if c in df.columns])

### Features

- `impressions_90d` — historical search visibility.
- `clicks_90d` — historical search interactions.
- `sessions_90d` — historical traffic.
- `ctr` — observed click-through rate.
- `avg_position` — observed search position; **0 means no data**, not rank zero.
- `content_age_days` and `days_since_last_update` — content freshness/age signals.
- `engagement_rate`, `scroll_rate`, `ai_traffic_pct` — observed engagement/referral signals.

### Label / proxy

- `is_declining_label` is the starter proxy, derived from `trend_direction == "down"`.
- For the stronger warehouse evaluation, the label should instead come from a later outcome window.

### Context

- `content_id` and `client_id` identify pages and clients and are used for grouping/splitting only.
- `content_type` and `main_intent` describe the page and can be retained as contextual fields.

### Excluded

- `trend_direction` and `trend_pct` are excluded from model features because the starter proxy is derived from them.
- Existing recommendation/decision flags such as `health_score`, `needs_ctr_fix`, `is_quick_win`, and `is_declining` are excluded because they encode prior rules or the outcome.

## 3. Verify it with queries/checks

*Every contract claim gets a check here. A contract claim without a check is a guess.*

In [ ]:
# Grain and basic counts
grain = df.groupby("content_id").size()
print("=== GRAIN ===")
print("Rows:", f"{len(df):,}")
print("Unique content IDs:", f"{grain.size:,}")
print("Content IDs with duplicates:", int((grain > 1).sum()))

# Missingness by field
print("\n=== MISSINGNESS ===")
check_cols = [c for c in feature_fields if c in df.columns]
missing = df[check_cols].isna().mean().mul(100).round(2).sort_values(ascending=False)
print(missing)

if "content_type" in df.columns and "word_count" in df.columns:
    print("\nMissing word_count by content type (%):")
    print((df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean() * 100).round(2)))

print("\n=== VALUE / WINDOW CHECKS ===")
for col in ["content_age_days", "days_since_last_update"]:
    if col in df.columns:
        print(col, "min=", df[col].min(), "max=", df[col].max())

## 4. Data limits

This starter snapshot is useful for building the refresh-scoring workflow, but it has important limits.

First, it has no report-date column, so the starter snapshot cannot by itself prove a future decline. The `is_declining_label` proxy is based on the current snapshot's `trend_direction`.

Second, the data is observational. Associations between page signals and performance do not prove that refreshing a page will cause traffic or ranking to improve.

Third, IDs are pseudonyms and should only be used for grouping, joining, and client-level validation.

Finally, missing values can follow content type. I will not blindly interpret every missing numeric value as zero; the final feature pipeline should preserve useful missingness indicators where appropriate.

## 5. Output

The contract supports a **ranked review queue**: each page receives a score that helps a content/SEO reviewer decide which pages to inspect first, with the underlying signals available as evidence.

The score is decision support; it is not an automatic instruction to refresh a page.

## Self-check

- [x] One row = one content page is stated and checked.
- [x] The trailing-90-day starter window is stated without inventing a report date.
- [x] Features, label/proxy, context, and excluded fields are separated.
- [x] IDs are not model features.
- [x] Label-derived fields are excluded.
- [x] Missingness and value checks are executed.
- [x] The starter proxy is clearly distinguished from a future warehouse outcome.
- [x] Output is a ranked refresh-review queue.